# Cerberus: Adversarial Training on Google Colab
## Full Defensive ML Implementation

This notebook trains robust models using adversarial examples.

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Navigate to Project & Install Dependencies

In [ ]:
import os
os.chdir('/content/drive/MyDrive/major_projekt')  # ADJUST THIS PATH
print(f'Working directory: {os.getcwd()}')
!ls -la

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install flask flask-cors matplotlib numpy scipy scikit-learn Pillow -q
!pip install -e ./cerberus -q
print('✓ Installed')

## Step 3: Verify GPU

In [ ]:
import torch
print(f'GPU Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
print(f'PyTorch: {torch.__version__}')

## Step 4: Train Robust Models (Adversarial Training)

In [ ]:
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from cerberus.dataset import get_cifar10_loaders
from cerberus.attacks import FSGMAttack

CONFIG = {
    'epochs': 15,
    'batch_size': 256,
    'lr': 0.01,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
}

print(f'Device: {CONFIG["device"]}')

def build_model(arch):
    if arch == 'resnet18':
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        model.fc = nn.Linear(512, 10)
    elif arch == 'resnet50':
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        model.fc = nn.Linear(2048, 10)
    elif arch == 'vgg16':
        model = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
        model.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        model.classifier = nn.Sequential(nn.Linear(512, 512), nn.ReLU(inplace=True), nn.Dropout(0.5), nn.Linear(512, 10))
    elif arch == 'mobilenetv2':
        model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        model.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(1280, 10))
    elif arch == 'efficientnetb0':
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        model.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(1280, 10))
    elif arch == 'densenet121':
        model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        model.classifier = nn.Linear(1024, 10)
    return model.to(CONFIG['device'])

def train_robust(arch, train_loader, val_loader):
    model = build_model(arch)
    criterion = nn.CrossEntropyLoss()
    opt = optim.SGD(model.parameters(), lr=CONFIG['lr'], momentum=0.9, weight_decay=5e-4)
    fgsm = FSGMAttack(model, eps=0.03)
    
    best_acc = 0
    for ep in range(1, CONFIG['epochs'] + 1):
        model.train()
        tc, tt = 0, 0
        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(CONFIG['device']), lbls.to(CONFIG['device'])
            
            # Generate adversarial examples
            with torch.no_grad():
                adv_imgs = fgsm.generate(imgs, lbls)
            
            opt.zero_grad()
            
            # Train on both clean and adversarial
            out_clean = model(imgs)
            loss_clean = criterion(out_clean, lbls)
            
            out_adv = model(adv_imgs)
            loss_adv = criterion(out_adv, lbls)
            
            loss = (loss_clean + loss_adv) / 2
            loss.backward()
            opt.step()
            
            _, pred = out_clean.max(1)
            tc += pred.eq(lbls).sum().item()
            tt += lbls.size(0)
        
        tacc = tc / tt * 100
        
        # Validate
        model.eval()
        vc, vt = 0, 0
        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs, lbls = imgs.to(CONFIG['device']), lbls.to(CONFIG['device'])
                out = model(imgs)
                _, pred = out.max(1)
                vc += pred.eq(lbls).sum().item()
                vt += lbls.size(0)
        
        vacc = vc / vt * 100
        print(f'[{arch:15s}] Ep {ep:2d}/15 | Train: {tacc:6.2f}% | Val: {vacc:6.2f}%')
        
        if vacc > best_acc:
            best_acc = vacc
            os.makedirs('models', exist_ok=True)
            torch.save(model.state_dict(), f'models/{arch}_robust_cifar10.pt')
            print(f'                   ✓ Saved ({vacc:.2f}%)')
    
    return best_acc

print('\nLoading CIFAR-10...')
train_loader, val_loader = get_cifar10_loaders('./data', batch_size=256, num_workers=2)
print('✓ Loaded')

In [ ]:
results = {}
for i, arch in enumerate(['resnet18', 'resnet50', 'vgg16', 'mobilenetv2', 'efficientnetb0', 'densenet121'], 1):
    print(f'\n[{i}/6] {arch.upper()} - ROBUST TRAINING')
    results[arch] = train_robust(arch, train_loader, val_loader)

print('\n' + '='*70)
print('ROBUST MODELS TRAINED')
print('='*70)
for arch, acc in results.items():
    print(f'{arch:20s}: {acc:6.2f}%')

## Step 5: Download Models

In [ ]:
# Models are automatically saved to your Google Drive
# Check: /content/drive/MyDrive/major_projekt/models/
print('✓ All robust models saved to Google Drive')
!ls -lh models/